# 04. ARIMA/SARIMA Model

В этом ноутбуке строим baseline и SARIMA-модель, оцениваем качество прогноза на последних 30 днях и сохраняем результаты.

## 1. Load processed data

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from statsmodels.tsa.statespace.sarimax import SARIMAX

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.metrics import evaluate_forecast
from src.preprocessing import train_test_split_time_series
from src.visualization import plot_train_test_forecast, save_plot

DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
IMAGES = PROJECT_ROOT / "images"
IMAGES.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(DATA_PROCESSED / "daily_sales_series.csv", parse_dates=["date"])
df_model = df[["date", "sales_clean"]].rename(columns={"sales_clean": "sales"})
display(df_model.head())

## 2. Train/test split

Последние 30 дней используются как test.

In [ ]:
train_df, test_df = train_test_split_time_series(df_model, test_days=30)

print("Train:", train_df["date"].min(), "-", train_df["date"].max(), train_df.shape)
print("Test:", test_df["date"].min(), "-", test_df["date"].max(), test_df.shape)

## 3. Baseline forecast

Baseline: прогноз равен средним продажам за последние 7 дней train-периода.

In [ ]:
baseline_value = train_df["sales"].tail(7).mean()
baseline_forecast = pd.DataFrame(
    {
        "date": test_df["date"].values,
        "forecast": baseline_value,
    }
)

display(baseline_forecast.head())

## 4. ARIMA/SARIMA model

Используем `SARIMAX` с умеренными параметрами и недельной сезонностью: `order=(1, 1, 1)`, `seasonal_order=(1, 1, 1, 7)`.

In [ ]:
train_series = train_df.set_index("date")["sales"].asfreq("D")

sarima_model = SARIMAX(
    train_series,
    order=(1, 1, 1),
    seasonal_order=(1, 1, 1, 7),
    enforce_stationarity=False,
    enforce_invertibility=False,
)

sarima_result = sarima_model.fit(disp=False, maxiter=100)
print(sarima_result.summary())

In [ ]:
sarima_pred = sarima_result.get_forecast(steps=len(test_df)).predicted_mean
sarima_forecast = pd.DataFrame(
    {
        "date": test_df["date"].values,
        "forecast": np.maximum(sarima_pred.to_numpy(), 0),
    }
)

display(sarima_forecast.head())

## 5. Forecast evaluation

In [ ]:
baseline_metrics = evaluate_forecast(test_df["sales"], baseline_forecast["forecast"], "Baseline 7-day mean")
sarima_metrics = evaluate_forecast(test_df["sales"], sarima_forecast["forecast"], "SARIMA")
metrics_df = pd.concat([baseline_metrics, sarima_metrics], ignore_index=True)

metrics_path = DATA_PROCESSED / "arima_metrics.csv"
metrics_df.to_csv(metrics_path, index=False)

forecast_output = pd.DataFrame(
    {
        "date": test_df["date"].values,
        "actual": test_df["sales"].values,
        "baseline_forecast": baseline_forecast["forecast"].values,
        "sarima_forecast": sarima_forecast["forecast"].values,
    }
)
forecast_path = DATA_PROCESSED / "arima_forecast.csv"
forecast_output.to_csv(forecast_path, index=False)

display(metrics_df.sort_values("WAPE"))
print(f"Saved metrics to: {metrics_path}")
print(f"Saved forecast to: {forecast_path}")

## 6. Forecast visualization

In [ ]:
plot_train_test_forecast(
    train_df.tail(180),
    test_df,
    sarima_forecast,
    target_col="sales",
    forecast_col="forecast",
    title="SARIMA Forecast vs Actual",
)
save_plot(IMAGES / "arima_forecast.png")
plt.show()

## 7. ARIMA conclusions

После запуска ноутбука перенесите значения MAE, RMSE, MAPE и WAPE в `reports/03_model_comparison.md`. SARIMA используется как статистическая модель с недельной сезонностью и будет сравниваться с Prophet в финальном ноутбуке.